# Checkpoint 3

Estimating slow conduction in cardiac tissue is a key inverse problem in cardiac electrophysiology. In this checkpoint, the goal is to reconstruct a **spatially varying slow-conduction field** from electrogram-derived activation times (see Checkpoint 1).

We use the **eikonal equation**, which models activation time as a function of spatial position and local conduction velocity. This formulation enables efficient forward simulations and is well suited for PDE-constrained optimization, physics-informed neural networks, and data-driven approaches.
The objective is to estimate a continuous conduction velocity map by minimizing the discrepancy between simulated and observed activation times, capturing both the location and severity of conduction slowing.

In this checkpoint 3, the anisotropy ratio will be fixed to 3.0 for all patients.

In [1]:
# imports

import numpy as np
import matplotlib.pyplot as plt


# Physics-based model implementation

Implemenatation of the algorithm presented in :

Sermesant, Maxime, et al. "An anisotropic multi-front fast marching method for real-time simulation of cardiac electrophysiology." Functional Imaging and Modeling of the Heart: 4th International Conference, FIHM 2007, Salt Lake City, UT, USA, June 7-9, 2007. Proceedings 4. Springer Berlin Heidelberg, 2007.

In [ ]:
# import numpy as np
# import heapq

# def anisotropic_FMM_GPT(
#     x0, y0, t0,
#     X, Y,
#     speed_input,
#     sigma_11, sigma_12, sigma_22,
#     N_edge=None
# ):
#     """
#     Fast-marching style solver with anisotropy and spatially varying speed.

#     Parameters
#     ----------
#     x0, y0 : float
#         Initial source location.
#     t0 : float
#         Initial activation time.
#     X, Y : 2D ndarray
#         Grid coordinates.
#     speed_input : float, callable, or 2D ndarray
#         Can be:
#         - a scalar: constant speed everywhere
#         - a function: speed_input(x, y)
#         - a 2D array with same shape as X, Y
#     sigma_11, sigma_12, sigma_22 : float
#         Components of the inverse of conductivity tensor.
#     N_edge : int or None
#         Optional. If None, inferred from grid shape.

#     Returns
#     -------
#     T_final : 2D ndarray
#         Arrival time map.
#     """

#     nrows, ncols = X.shape
#     if N_edge is None:
#         N_edge = nrows - 1

#     INF = 1e4
#     P = np.linspace(0.0, 1.0, 11)

    # nearest grid node to source
#     j0 = np.abs(X[0, :] - x0).argmin()
#     i0 = np.abs(Y[:, 0] - y0).argmin()

#     T = np.full((nrows, ncols), INF, dtype=float)
#     T_final = np.full((nrows, ncols), INF, dtype=float)

#     T[i0, j0] = t0
#     T_final[i0, j0] = t0

#     known = np.zeros((nrows, ncols), dtype=bool)

    # min-heap of (arrival_time, i, j)
#     heap = [(t0, i0, j0)]

#     def get_speed(i, j, x, y):
#         if callable(speed_input):
#             val = speed_input(x, y)
#         elif np.isscalar(speed_input):
#             val = speed_input
#         else:
#             val = speed_input[i, j]

#         if val <= 0:
#             raise ValueError(f"Speed must be positive. Got {val} at ({i}, {j}).")
#         return val

#     def metric_dist(xa, ya, xb, yb, speed_val):
#         dx = xa - xb
#         dy = ya - yb
#         return np.sqrt(
#             sigma_11 * dx * dx +
#             2.0 * sigma_12 * dx * dy +
#             sigma_22 * dy * dy
#         ) / speed_val

#     def update_fun(
#         T_xi, T_x, T_y,
#         X_xi, X_x, X_y,
#         Y_xi, Y_x, Y_y,
#         flag_known,
#         i_xi, j_xi
#     ):
#         speed_val = get_speed(i_xi, j_xi, X_xi, Y_xi)

#         if flag_known:
#             dx = P * (X_xi - X_x) + (1.0 - P) * (X_xi - X_y)
#             dy = P * (Y_xi - Y_x) + (1.0 - P) * (Y_xi - Y_y)

#             triangle_vals = (
#                 P * T_x + (1.0 - P) * T_y
#                 + np.sqrt(
#                     sigma_11 * dx * dx +
#                     2.0 * sigma_12 * dx * dy +
#                     sigma_22 * dy * dy
#                 ) / speed_val
#             )
#             cand = triangle_vals.min()
#         else:
#             cand = T_x + metric_dist(X_xi, Y_xi, X_x, Y_x, speed_val)

#         return min(T_xi, cand)

    # side nodes used for triangle updates
#     side_map = {
#         (-1,  0): [(-1, +1), (-1, -1)],  # north
#         (+1,  0): [(+1, +1), (+1, -1)],  # south
#         ( 0, -1): [(+1, -1), (-1, -1)],  # west
#         ( 0, +1): [(+1, +1), (-1, +1)],  # east
#     }

#     while heap:
#         t_sel, i_sel, j_sel = heapq.heappop(heap)

        # ignore stale entries
#         if known[i_sel, j_sel]:
#             continue
#         if t_sel > T_final[i_sel, j_sel]:
#             continue

#         known[i_sel, j_sel] = True

        # visit 4-neighbors
#         for di, dj in [(-1, 0), (1, 0), (0, -1), (0, +1)]:
#             ni, nj = i_sel + di, j_sel + dj

#             if not (0 <= ni < nrows and 0 <= nj < ncols):
#                 continue
#             if known[ni, nj]:
#                 continue

#             new_T = INF

#             for sdi, sdj in side_map[(di, dj)]:
#                 si, sj = i_sel + sdi, j_sel + sdj

#                 if not (0 <= si < nrows and 0 <= sj < ncols):
#                     continue

#                 new_T = update_fun(
#                     T_xi=new_T,
#                     T_x=T_final[i_sel, j_sel],
#                     T_y=T_final[si, sj],
#                     X_xi=X[ni, nj], X_x=X[i_sel, j_sel], X_y=X[si, sj],
#                     Y_xi=Y[ni, nj], Y_x=Y[i_sel, j_sel], Y_y=Y[si, sj],
#                     flag_known=known[si, sj],
#                     i_xi=ni, j_xi=nj
#                 )

#             if new_T < T_final[ni, nj]:
#                 T[ni, nj] = new_T
#                 T_final[ni, nj] = new_T
#                 heapq.heappush(heap, (new_T, ni, nj))

#     return T_final

# Eikonal equation

Let us consider the Eikonal model: find T(x,y)

\begin{aligned}
  & 80 \sqrt{ \nabla T(x,y) D \nabla T(x,y)} = 1  \,, \quad (x,y) \in [-2,2] \times [-2,2]\,,\\
  & u(x,y) = t_0,t_1  \,, \qquad \qquad \qquad \qquad \qquad x=x_0,x_1,y=y_0,y_1 \,, \\
\end{aligned}

where 
\begin{aligned}
D = 1\ \vec{b} \otimes \vec{b} + \frac{1}{a_{ratio}}  \ \vec{a} \otimes \vec{a},
\end{aligned}
being $ \vec{b} $ the given fiber orientation and $ \vec{a} $ the given cross-fiber orientation (orthogonal to $ \vec{b}$).

The unknown parameters of the problem are: 
- conduction velocity (tissue property common to all the patients),
- anisotropy ratio $a_{ratio}$ (tissue property common to all the patients),
- the slow conduction positions $(x_{SC},y_{SC})$ and its radius $r_{SC}$.

In [ ]:
# X_mesh, Y_mesh = np.meshgrid(np.linspace(-2.0,2.0,401), np.linspace(-2.0,2.0,401))

# x0 = -1.5
# y0 = -1

# theta_fiber = 0
# a_ratio = 5

# theta0 = np.pi/2 - theta_fiber
# a = np.array([np.cos(theta0), np.sin(theta0)]).T
# b = np.array([np.cos(theta0-np.pi/2), np.sin(theta0-np.pi/2)]).T

# D = ( (1/a_ratio)*np.tensordot( a, a,  axes=0) + np.tensordot( b, b,  axes=0)  )
# print(D)

# D_inv = np.linalg.inv(D)
# sigma_11 = D_inv[0][0]
# sigma_12 = D_inv[0][1]
# sigma_21 = D_inv[1][0]
# sigma_22 = D_inv[1][1]

# xc, yc = 0.5, 0.5
# r = 0.5

# speed_field = lambda x, y: 20.0 if (x - xc)**2 + (y - yc)**2 <= r**2 else 80.0

# activation_time = anisotropic_FMM_GPT( x0 , y0 , 0, X_mesh, Y_mesh, speed_field, sigma_11,sigma_12, sigma_22, N_edge=200 )




In [ ]:

# plt.contour(X_mesh, Y_mesh, activation_time*1000 , 10) 
# plt.gca().set_aspect(1)
# plt.colorbar()

In [ ]:
# N_samples = 1

# np.random.seed(32)

# measurements_dataset = []
# groundtruth_dataset = []

# for i_sample in range(N_samples):

#     x0 = np.random.uniform(low=1.5,high=1.5)
#     y0 = np.random.uniform(low=-1.5,high=1.5)
#     x1 = np.random.uniform(low=1.5,high=1.5)
#     y1 = np.random.uniform(low=-1.5,high=1.5)
#     theta_fiber = 0.0
#     a_ratio = np.random.uniform(low=1,high=9)

#     theta0 = np.pi/2 - theta_fiber
#     a = np.array([np.cos(theta0), np.sin(theta0)]).T
#     b = np.array([np.cos(theta0-np.pi/2), np.sin(theta0-np.pi/2)]).T

#     D = ( (1/a_ratio)*np.tensordot( a, a,  axes=0) + np.tensordot( b, b,  axes=0)  )
#     print(D)

#     D_inv = np.linalg.inv(D)
#     sigma_11 = D_inv[0][0]
#     sigma_12 = D_inv[0][1]
#     sigma_21 = D_inv[1][0]
#     sigma_22 = D_inv[1][1]

    # compute activation time
#     activation_time = anisotropic_FMM_GPT( x0 , y0 , 0.0, X_mesh, Y_mesh, speed_field, sigma_11,sigma_12, sigma_22, N_edge=200 )

    # rescaling in ms
#     activation_time = activation_time*1e3

    # measurements location
#     Spline_spacing = [0.5, 0.7, 1.3, 1.5]
#     Angles_spacing =  [np.pi/10.0+0, np.pi/10.0+(2)*np.pi/5, np.pi/10.0+(4)*np.pi/5, np.pi/10.0+(6)*np.pi/5, np.pi/10.0+(8)*np.pi/5 ]
    
#     index_extracted = []
#     x_meas = []
#     y_meas = []
#     time_meas = []

#     for i in range(4):
        #print(i)
#         for j in range(5):
            #print(j)
#             MeasMat = (X_mesh - Spline_spacing[i]*np.cos(Angles_spacing[j]) )**2 + (Y_mesh - Spline_spacing[i]*np.sin(Angles_spacing[j]) )**2
#             [i_sel, j_sel] = np.unravel_index(MeasMat.argmin(), MeasMat.shape) 
#             index_extracted.append(i_sel)
#             x_meas.append(X_mesh[i_sel,j_sel])
#             y_meas.append(Y_mesh[i_sel,j_sel])
#             time_meas.append(activation_time[i_sel,j_sel])

#     measurements_dataset.append( [ x_meas, y_meas, time_meas ] )
#     groundtruth_dataset.append([activation_time])

In [ ]:
# plt.figure(1)
# plt.scatter( x_meas, y_meas, c=time_meas, vmin= 0, vmax=np.max(activation_time) )
# plt.contour(X_mesh, Y_mesh, activation_time , 10)
# plt.gca().set_aspect(1)
# plt.colorbar()

# Checkpoint 3

Implement your algorithm to process the traces of the 20 unipolar electrogram signals and identify the center and radius of the slow conduction region.

The algorithm must return the following outputs:

1. **slow conduction fields evaluated on a 16x16 grid** 


Provide your algorithm's implementation in this notebook usign the signature defined below.

In [ ]:
# load data
# signals = np.load('checkpoint3dataset.npy')

# select event to be display
# ind_event = 4

# signals_event = signals[ind_event,:,:]

In [ ]:
# select a spline (options: 0,1,2,3,4)
# ind_spline = 0
# plot signals

# plt.figure(1+ind_spline)
# countEL = 1
# for indS in np.arange(ind_spline*4,(ind_spline+1)*4,1):
#     plt.subplot(4,1,countEL)
            
#     plt.plot(signals[ind_event,indS,:])
#     plt.grid(True)

#     countEL = countEL + 1

# plt.show()

In [ ]:
# def load_and_plot_velocity_fields(
#     filename,
#     sim_idx=0,
#     N=None,
#     M=None,
#     h=None
# ):
#     velocity_fields = np.load(filename)

#     field16 = velocity_fields[sim_idx]

#     if N is not None and M is not None and h is not None:
#         step_x = N // 16
#         step_y = M // 16

#         ix = np.arange(0, step_x * 16, step_x)
#         iy = np.arange(0, step_y * 16, step_y)

#         Lx_sub = ix[-1] * h
#         Ly_sub = iy[-1] * h

#         extent = [0, Lx_sub, 0, Ly_sub]
#     else:
#         extent = None

#     plt.figure(figsize=(5, 4))
#     plt.imshow(
#         field16.T,
#         origin='lower',
#         extent=extent,
#         cmap='viridis',
#         aspect='equal'
#     )
#     plt.colorbar(label='Conduction scale')
#     plt.title(f'Case {sim_idx} - velocity field (16x16)')
#     plt.xlabel('x')
#     plt.ylabel('y')
#     plt.tight_layout()
#     plt.show()

In [ ]:
# N = np.int32(2*164)
# M = np.int32(2*164)
# h = 4.0/N
# load_and_plot_velocity_fields("checkpoint3velocityfield16.npy", sim_idx=2,N=N,M=M,h=h)


In [11]:
# =====================================================================
# Checkpoint 3 - first-version autodecoder + DATA AUGMENTATION
# =====================================================================
# Same architecture as the first version (per-scenario latent Embed +
# coordinate-MLP decoder + encoder MLP). The ONLY change is that the model is
# TRAINED on AUGMENTED data built from the 25 real scenarios (no synthetic data,
# no FMM). Run the next cells in order: they train the model live (~6-7 min),
# then define GroupSolutionCP3.

import pickle, time
import numpy as onp
import jax, jax.numpy as jnp, optax
import jax.tree_util as jtu
from flax import nnx
from typing import Callable, Optional, Sequence

jax.config.update("jax_enable_x64", True)


class CustomMLP(nnx.Module):
    def __init__(self, in_features, out_features, rngs, hidden_features,
                 activation_function=nnx.tanh, final_activation_function=None):
        super().__init__()
        self.activation_function = activation_function
        self.final_activation_function = final_activation_function
        sizes = [in_features] + list(hidden_features) + [out_features]
        self.n_layers = len(sizes) - 1
        for idx in range(self.n_layers):
            setattr(self, f'layer_{idx}', nnx.Linear(
                sizes[idx], sizes[idx + 1],
                kernel_init=nnx.initializers.lecun_normal(),
                bias_init=nnx.initializers.zeros_init(),
                dtype=jnp.float64, param_dtype=jnp.float64, rngs=rngs))

    def __call__(self, x):
        for idx in range(self.n_layers):
            x = getattr(self, f'layer_{idx}')(x)
            if idx < self.n_layers - 1:
                x = self.activation_function(x)
        if self.final_activation_function is not None:
            x = self.final_activation_function(x)
        return x


class AutoDecoder(nnx.Module):
    def __init__(self, embedder, decoder):
        super().__init__(); self.embedder = embedder; self.decoder = decoder

    def __call__(self, points, indices):
        z = self.embedder(indices.astype(jnp.int32))
        return self.decoder(jnp.concatenate([points, z], axis=-1))


## Data augmentation

We only have **25 real scenarios**, which is too few for the encoder to generalise.
We enlarge the training set using **exact symmetries of the physical problem**, so
the augmented samples are *real* (no synthetic forward model, no sim-to-real gap):

1. **x → −x mirror.** The fibres lie along *x* and the anisotropy tensor is
   `diag(1, 3)`, which is invariant under `x → −x`. The 20 electrodes are placed
   symmetrically about the *y*-axis, so a reflection just **permutes** them
   (`PERM`). Therefore, for every scenario, `(field[:, ::-1], ATs[PERM])` is a
   genuinely new, physically valid example with the slow zone on the opposite
   side → it **doubles** the field set (25 → 50) and widens the coverage of slow-zone positions.

2. **Beat swap.** Each scenario contains two activation fronts from two focal
   points; the conduction field is *static*, so the two beats are interchangeable.
   Feeding the encoder `[AT2, AT1]` with the same latent target teaches it to be
   robust to beat order.

3. **Activation-time jitter.** Small Gaussian noise (±1.5 ms) added to the ATs,
   to make the encoder robust to measurement/extraction noise.

The **autodecoder** is trained on the 50 mirrored fields (one latent each); the
**encoder** is trained on the resulting `(ATs → latent)` pairs after applying
mirror + beat-swap + noise (≈ 500 pairs instead of 25). Architecture, losses and
hyper-parameters are **identical** to the first version.

In [ ]:
# ---- load the 25 real scenarios -------------------------------------------------
# vf  = onp.concatenate([onp.load('checkpoint3velocityfield16.npy'),
#                        onp.load('checkpoint3velocityfield16_p2.npy')])
# sig = onp.concatenate([onp.load('checkpoint3dataset.npy'),
#                        onp.load('checkpoint3dataset_p2.npy')])
# sta = onp.concatenate([onp.load('checkpoint3starting.npy'),
#                        onp.load('checkpoint3starting_p2.npy')])
# N_REAL = len(vf)

# electrode permutation under x -> -x  (signals are in the checkpoint1 order)
# _SENS = onp.load('checkpoint1sensorlocations.npy') - 2.0
# PERM = onp.array([int(onp.argmin(onp.sum((_SENS - onp.array([-_SENS[k, 0], _SENS[k, 1]])) ** 2, axis=1)))
#                   for k in range(20)])

# _T_AXIS = onp.linspace(0, 500, 501)


# def _extract_AT_all(signals_event, t_ref, window_ms=120.0):
#     """Activation time (ms) at all 20 electrodes: steepest -dV/dt within +/-window_ms."""
#     ATs = []
#     for e in range(20):
#         dv = onp.gradient(signals_event[e], _T_AXIS)
#         mask = (_T_AXIS >= max(0, t_ref - window_ms)) & (_T_AXIS <= min(500, t_ref + window_ms))
#         ATs.append(_T_AXIS[mask][onp.argmin(dv[mask])])
#     return onp.array(ATs)


# ---- AUGMENTATION 1: x -> -x mirror  ->  25 + 25 = 50 fields for the autodecoder
# fields = onp.concatenate([vf, vf[:, :, ::-1]], axis=0)
# N_TRAIN = len(fields)

# g = onp.linspace(-2, 2, 16)
# gx, gy = onp.meshgrid(g, g, indexing='ij')
# grid_norm = (onp.stack([gx.ravel(), gy.ravel()], axis=1) + 2.0) / 4.0    # (256, 2) in [0,1]
# points_train  = jnp.array(onp.concatenate([grid_norm] * N_TRAIN, axis=0))
# targets_train = jnp.array(onp.concatenate([fields[i].ravel() for i in range(N_TRAIN)])[:, None])
# indices_train = jnp.array(onp.concatenate([onp.full(256, i, dtype=onp.int32) for i in range(N_TRAIN)]))

# print(f'Real scenarios          : {N_REAL}')
# print(f'Augmented (mirrored) set: {N_TRAIN} fields for the autodecoder')


In [ ]:
# ---- train the AUTODECODER (identical recipe to the first version) --------------
# LATENT_DIM, HIDDEN_DEC = 8, [64, 64, 64]
# REG_WEIGHT, NUM_ADAM, LR_ADAM = 1e-3, 15000, 3e-3      # set NUM_ADAM lower for a quick test

# embedder = nnx.Embed(num_embeddings=N_TRAIN, features=LATENT_DIM, rngs=nnx.Rngs(42),
#                      embedding_init=nnx.initializers.normal(stddev=0.01))
# decoder = CustomMLP(2 + LATENT_DIM, 1, nnx.Rngs(43), HIDDEN_DEC, nnx.tanh, nnx.sigmoid)
# autodecoder = AutoDecoder(embedder, decoder)


# @nnx.jit
# def loss_fn(m, p, i, t):
#     pred = m(p, i).squeeze()
#     rmse = jnp.sqrt(jnp.mean(jnp.square(pred - t.squeeze())))
#     z = m.embedder(i.astype(jnp.int32))
#     return rmse + REG_WEIGHT * jnp.mean(jnp.sum(jnp.square(z), axis=1)), rmse


# @nnx.jit
# def train_step(opt, m, p, i, t):
#     (v, r), gr = nnx.value_and_grad(loss_fn, has_aux=True)(m, p, i, t)
#     opt.update(m, gr)
#     return v, r


# opt = nnx.Optimizer(autodecoder, optax.adam(LR_ADAM, b1=0.99), wrt=nnx.Param)
# print('Training autodecoder ...')
# _t = time.time()
# for epoch in range(NUM_ADAM):
#     _, r = train_step(opt, autodecoder, points_train, indices_train, targets_train)
#     if (epoch + 1) % 1500 == 0:
#         print(f'  step {epoch+1:5d}/{NUM_ADAM}   recon RMSE = {float(r):.4e}   ({time.time()-_t:.0f}s)')
# print(f'Autodecoder trained in {time.time()-_t:.0f}s.')

# the learned latent code of each (augmented) field
# Z = onp.array(autodecoder.embedder.embedding.value)   # (N_TRAIN, LATENT_DIM)


In [ ]:
# ---- build AUGMENTED encoder data (ATs -> latent) -------------------------------
# augmentation 2 = beat swap, augmentation 3 = activation-time jitter
# NOISE_MS, N_NOISE = 1.5, 4
# _rng = onp.random.default_rng(0)
# enc_in, enc_tg = [], []

# def _add(a1, a2, z):
#     enc_in.append(onp.concatenate([a1, a2]) / 500.0)
#     enc_tg.append(z)

# for i in range(N_REAL):
#     t0_, t1_ = sta[i]
#     AT1 = _extract_AT_all(sig[i], t0_)
#     AT2 = _extract_AT_all(sig[i], t1_)
#     z_real, z_mirror = Z[i], Z[i + N_REAL]          # latent of the field and of its mirror

#     _add(AT1, AT2, z_real)                            # identity
#     _add(AT2, AT1, z_real)                            # beat swap
#     _add(AT1[PERM], AT2[PERM], z_mirror)             # mirror
#     _add(AT2[PERM], AT1[PERM], z_mirror)             # mirror + beat swap
#     for _ in range(N_NOISE):                          # + jitter
#         n1, n2 = AT1 + _rng.normal(0, NOISE_MS, 20), AT2 + _rng.normal(0, NOISE_MS, 20)
#         _add(n1, n2, z_real);            _add(n2, n1, z_real)
#         _add(n1[PERM], n2[PERM], z_mirror); _add(n2[PERM], n1[PERM], z_mirror)

# enc_inputs = jnp.array(onp.stack(enc_in)); enc_targets = jnp.array(onp.stack(enc_tg))
# print(f'Encoder training pairs (augmented): {enc_inputs.shape[0]}  (vs {N_REAL} without augmentation)')

# ---- train the ENCODER (identical recipe to the first version) ------------------
# HIDDEN_ENC, NUM_ENC, LR_ENC = [64, 64], 10000, 1e-3
# encoder_mlp = CustomMLP(40, LATENT_DIM, nnx.Rngs(99), HIDDEN_ENC, nnx.tanh, None)


# @nnx.jit
# def enc_loss(m, x, z):
#     return jnp.mean(jnp.sum(jnp.square(m(x) - z), axis=1))


# @nnx.jit
# def enc_step(opt, m, x, z):
#     l, gr = nnx.value_and_grad(enc_loss)(m, x, z)
#     opt.update(m, gr)
#     return l


# enc_opt = nnx.Optimizer(encoder_mlp, optax.adam(LR_ENC), wrt=nnx.Param)
# print('Training encoder ...')
# _t = time.time()
# for epoch in range(NUM_ENC):
#     l = enc_step(enc_opt, encoder_mlp, enc_inputs, enc_targets)
# print(f'Encoder trained: loss={float(l):.4e}  ({time.time()-_t:.0f}s)')

# ---- expose the trained models to GroupSolutionCP3 (and save a checkpoint) ------
# _ENC, _DEC, _GRID_NORM = encoder_mlp, autodecoder.decoder, jnp.array(grid_norm)
# _CKPT = True


# def _flax_state_to_numpy(state):
#     leaves, treedef = jtu.tree_flatten(state)
#     return {'leaves': [onp.array(l) for l in leaves], 'treedef': treedef}

# with open('autodecoder_aug.pkl', 'wb') as _f:
#     pickle.dump({'dec_params': _flax_state_to_numpy(nnx.state(autodecoder)),
#                  'enc_params': _flax_state_to_numpy(nnx.state(encoder_mlp)),
#                  'latent_dim': LATENT_DIM, 'hidden_dec': HIDDEN_DEC, 'hidden_enc': HIDDEN_ENC,
#                  'grid_norm': onp.array(grid_norm), 'n_train': N_TRAIN}, _f)
# print('Saved autodecoder_aug.pkl')


In [ ]:
# ============================================================================
# INFERENCE SETUP  -  load the trained encoder+decoder from the saved checkpoint
# Run this cell, then run GroupSolutionCP3 below. No training required (<1 s).
# ============================================================================
import pickle
import jax.tree_util as jtu

_T_AXIS = onp.linspace(0, 500, 501)


def _extract_AT_all(signals_event, t_ref, window_ms=120.0):
    """Activation time (ms) at all 20 electrodes: steepest -dV/dt within +/-window_ms."""
    ATs = []
    for e in range(20):
        dv = onp.gradient(signals_event[e], _T_AXIS)
        mask = (_T_AXIS >= max(0, t_ref - window_ms)) & (_T_AXIS <= min(500, t_ref + window_ms))
        ATs.append(_T_AXIS[mask][onp.argmin(dv[mask])])
    return onp.array(ATs)


def _restore_models(ckpt):
    d, h_dec, h_enc, n_train = (ckpt['latent_dim'], ckpt['hidden_dec'],
                                ckpt['hidden_enc'], ckpt['n_train'])
    rngs_ = nnx.Rngs(0)

    # decoder (rebuilt inside the full AutoDecoder, then extracted)
    emb_ = nnx.Embed(num_embeddings=n_train, features=d, rngs=rngs_)
    dec_ = CustomMLP(2 + d, 1, rngs_, h_dec, nnx.tanh, nnx.sigmoid)
    full = AutoDecoder(embedder=emb_, decoder=dec_)
    raw = ckpt['dec_params']
    nnx.update(full, raw['treedef'].unflatten([jnp.array(l) for l in raw['leaves']]))
    dec_ = full.decoder

    # encoder
    enc_ = CustomMLP(40, d, rngs_, h_enc, nnx.tanh, None)
    raw = ckpt['enc_params']
    nnx.update(enc_, raw['treedef'].unflatten([jnp.array(l) for l in raw['leaves']]))
    return enc_, dec_


with open('autodecoder_aug.pkl', 'rb') as _f:
    _ckpt = pickle.load(_f)
_ENC, _DEC = _restore_models(_ckpt)
_GRID_NORM = jnp.array(_ckpt['grid_norm'])
print('Loaded trained encoder + decoder from autodecoder_aug.pkl')


In [15]:
def GroupSolutionCP3(signals_event, x_0, y_0, t_0, x_1, y_1, t_1):
    """
    Estimate the 16x16 slow-conduction velocity field from the 20 unipolar signals.

    Pipeline (first-version autodecoder, trained with data augmentation):
        signals -> 40 activation times -> encoder -> latent z* -> decoder -> field16
    """
    signals_event = onp.asarray(signals_event, dtype=float)

    # 1. activation times for both beats
    AT1 = _extract_AT_all(signals_event, t_0)
    AT2 = _extract_AT_all(signals_event, t_1)
    feat = jnp.array(onp.concatenate([AT1, AT2]) / 500.0)[None, :]   # (1, 40)

    # 2. encoder -> latent code
    z_star = _ENC(feat)                                             # (1, latent_dim)

    # 3. decoder -> conduction map on the 16x16 grid
    x_in = jnp.concatenate([_GRID_NORM, jnp.tile(z_star, (256, 1))], axis=-1)
    field = onp.array(_DEC(x_in)).ravel()
    return onp.clip(field, 0.0, 1.0).reshape(16, 16)


# Test

Quick check of `GroupSolutionCP3` on the 25 available scenarios (ground truth vs
prediction). Same architecture as the first version, only the training data is augmented.

In [ ]:
# import numpy as np
# import time

# _sig = np.concatenate([np.load('checkpoint3dataset.npy'),     np.load('checkpoint3dataset_p2.npy')])
# _foc = np.concatenate([np.load('checkpoint3FocalPoints.npy'), np.load('checkpoint3FocalPoints_p2.npy')])
# _sta = np.concatenate([np.load('checkpoint3starting.npy'),    np.load('checkpoint3starting_p2.npy')])
# _gt  = np.concatenate([np.load('checkpoint3velocityfield16.npy'), np.load('checkpoint3velocityfield16_p2.npy')])
# _n = len(_sig)

# preds = np.zeros_like(_gt)
# _t0 = time.time()
# for i in range(_n):
#     preds[i] = GroupSolutionCP3(_sig[i], _foc[i, 0, 0], _foc[i, 0, 1], _sta[i, 0],
#                                 _foc[i, 1, 0], _foc[i, 1, 1], _sta[i, 1])
# dt = (time.time() - _t0) / _n

# mae = np.mean([np.mean(np.abs(preds[i] - _gt[i])) for i in range(_n)])
# rmse = np.mean([np.sqrt(np.mean((preds[i] - _gt[i]) ** 2)) for i in range(_n)])
# print('In-sample over %d scenarios: MAE=%.4f  RMSE=%.4f  (%.3f s / case)' % (_n, mae, rmse, dt))


In [ ]:
# import matplotlib.pyplot as plt

# _cases = [0, 2, 5, 9, 11, 19]
# fig, axes = plt.subplots(len(_cases), 3, figsize=(9, 3 * len(_cases)))
# for row, i in enumerate(_cases):
#     gtf, prd = _gt[i], preds[i]
#     err = np.abs(prd - gtf)
#     panels = [
#         (gtf, 'GT  (case %d)' % i,                                  'viridis', (0.2, 1.0)),
#         (prd, 'Pred  RMSE=%.3f' % np.sqrt(np.mean((prd - gtf) ** 2)), 'viridis', (0.2, 1.0)),
#         (err, '|error|',                                            'magma',   (0.0, 0.4)),
#     ]
#     for col, (data, title, cmap, vlim) in enumerate(panels):
#         ax = axes[row, col]
#         im = ax.imshow(data.T, origin='lower', extent=[-2, 2, -2, 2],
#                        cmap=cmap, vmin=vlim[0], vmax=vlim[1], aspect='equal')
#         ax.set_title(title, fontsize=9)
#         ax.set_xticks([]); ax.set_yticks([])
#         fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
# fig.suptitle('First-version autodecoder + data augmentation: GT vs prediction', y=1.001)
# fig.tight_layout()
# plt.show()
